# 03 — Inspect training tiles

Catch bugs **before** burning compute. This notebook builds the same
`TileDataset` that `train.py` uses and renders a few `(image, mask)` samples
straight from `__getitem__`. If the panels look right here, the trainer is
seeing the same thing.

Checks:

1. Image tile + GT mask align — line pixels in the mask sit on top of line
   pixels in the image.
2. Augmentation is sane — flipped/rotated tiles stay paired.
3. Foreground ratio per tile is in the expected 1–5% range (positive-centered
   sampling should lift it well above the 0.1–0.8% whole-image average).
4. Normalization stats look reasonable (mean ≈ 0 ± 1 per channel after
   ImageNet normalization).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

print("cwd:", Path.cwd())

import yaml
import numpy as np
import matplotlib.pyplot as plt

from data.augmentation import TileAugmenter
from data.dataset import TileDataset

## Build the dataset (same params as `train.py`)

In [ ]:
base_cfg  = yaml.safe_load(open("configs/base.yaml"))
train_cfg = yaml.safe_load(open("configs/train.yaml"))

# Shallow merge same as train.py.
cfg = dict(base_cfg)
for k, v in train_cfg.items():
    if isinstance(v, dict) and isinstance(cfg.get(k), dict):
        cfg[k] = {**cfg[k], **v}
    else:
        cfg[k] = v

dataset_root = (Path.cwd() / cfg["data"]["dataset_root"]).resolve()
train_dir = dataset_root / cfg["data"]["train_dir"]

aug = TileAugmenter(
    hflip_prob=cfg["augmentation"]["hflip_prob"],
    vflip_prob=cfg["augmentation"]["vflip_prob"],
    rotate_90_prob=cfg["augmentation"]["rotate_90_prob"],
    seed=0,
)

train_ds = TileDataset(
    split_dir=train_dir,
    tile_size=cfg["data"]["tile_size"],
    stride=cfg["data"]["stride"],
    mode=cfg["data"]["train_mode"],
    merge_radius=cfg["polyline"]["merge_radius"],
    thickness=cfg["rasterize"]["thickness"],
    augmenter=aug,
    mean=tuple(cfg["normalization"]["mean"]),
    std=tuple(cfg["normalization"]["std"]),
    samples_per_epoch_per_image=cfg["data"].get("samples_per_epoch_per_image", 8),
    seed=0,
)
print(f"train tiles per epoch: {len(train_ds)}")
print(f"tile_size:            {cfg['data']['tile_size']}")

## Render a few samples

Each row: original tile (de-normalized), GT mask, overlay (mask in red on top of the tile). Foreground ratio is printed per sample — should be roughly 1–5% for positive-centered tiles.

In [ ]:
N_SAMPLES = 4

mean = np.array(cfg["normalization"]["mean"], dtype=np.float32).reshape(3, 1, 1)
std  = np.array(cfg["normalization"]["std"],  dtype=np.float32).reshape(3, 1, 1)


def denormalize(img_t):
    img = img_t.numpy() * std + mean         # [3, H, W]
    img = np.clip(img, 0, 1).transpose(1, 2, 0)
    return (img * 255).astype(np.uint8)


def overlay(rgb, mask_2d, color=(255, 0, 0), alpha=0.55):
    out = rgb.copy()
    sel = mask_2d > 0.5
    for c in range(3):
        out[..., c] = np.where(
            sel,
            (1 - alpha) * out[..., c] + alpha * color[c],
            out[..., c],
        ).astype(np.uint8)
    return out


fig, axes = plt.subplots(N_SAMPLES, 3, figsize=(18, 6 * N_SAMPLES))
for i in range(N_SAMPLES):
    sample = train_ds[i]
    img = denormalize(sample.image)             # [H, W, 3] uint8 RGB
    msk = sample.mask[0].numpy()                # [H, W] float in {0, 1}
    fg_ratio = float(msk.mean())                # fraction of foreground pixels

    axes[i][0].imshow(img)
    axes[i][0].set_title(
        f"sample {i}  img_id={sample.img_id}  origin={sample.tile_origin}",
        fontsize=10,
    )
    axes[i][0].set_axis_off()

    axes[i][1].imshow(msk, cmap="gray", vmin=0, vmax=1)
    axes[i][1].set_title(f"GT mask  --  foreground {100*fg_ratio:.2f}%", fontsize=10)
    axes[i][1].set_axis_off()

    axes[i][2].imshow(overlay(img, msk))
    axes[i][2].set_title("overlay (mask in red)", fontsize=10)
    axes[i][2].set_axis_off()

fig.tight_layout()
plt.show()

## Aggregate statistics over many samples

A robust quick-look at:

- the **distribution of foreground pixel ratios** across many random tiles, and
- the **per-channel normalized-image statistics** (should be roughly centered).

In [ ]:
N_AGG = 64

fg_ratios = []
img_means = []   # per-sample, per-channel means (post-normalization)
img_stds  = []
for k in range(N_AGG):
    s = train_ds[k]
    fg_ratios.append(float(s.mask.mean()))
    arr = s.image.numpy()
    img_means.append(arr.reshape(3, -1).mean(axis=1))
    img_stds.append(arr.reshape(3, -1).std(axis=1))

fg_ratios = np.array(fg_ratios)
img_means = np.stack(img_means)
img_stds  = np.stack(img_stds)

print(f"foreground %:")
print(f"  min={100*fg_ratios.min():.3f}  "
      f"median={100*np.median(fg_ratios):.3f}  "
      f"mean={100*fg_ratios.mean():.3f}  "
      f"max={100*fg_ratios.max():.3f}")
print()
print(f"per-channel image mean (normalized, n={N_AGG}):  {img_means.mean(axis=0)}")
print(f"per-channel image std  (normalized, n={N_AGG}):  {img_stds.mean(axis=0)}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(100 * fg_ratios, bins=20, edgecolor="k")
ax.set_xlabel("foreground % per tile")
ax.set_ylabel("count")
ax.set_title(f"Foreground distribution across {N_AGG} positive-centered tiles")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()